# Week 1 Tue: Functions, control flow, probability intuition, and expected value

Estimated time: 4 hours

## Why this matters
Quant code is built from reusable functions, and quant decisions often reduce to expected value under uncertainty.

## Study blocks
- Block 1 (45 min): Review Python functions and reusable finance logic.
- Block 2 (60 min): Learn control flow and payoff rules.
- Block 3 (60 min): Build probability intuition and expected value.
- Block 4 (45 min): Connect payoff asymmetry to trading setups.
- Block 5 (30 min): Code, reflect, and interview drill.

## Concept notes
### Functions as reusable research building blocks
A function is a named block of logic that you can call again with different inputs. This helps keep research clean, testable, and less error-prone.

Technical note: A function maps inputs to outputs. In quant work, small functions for returns, drawdowns, expected value, or signal transforms keep the pipeline modular.

Finance example: Instead of recomputing a return formula manually in many cells, a function lets you standardize the calculation across assets.

### Control flow as rule logic
Control flow decides what the program should do under different conditions.

Technical note: Conditionals like if/else mirror rule-based finance logic: if a signal crosses a threshold, enter; otherwise stay flat.

Finance example: A stop-loss or risk cap is basically a decision rule embedded in code.

### Expected value and asymmetric payoffs
Expected value is the average outcome if the same uncertain setup is repeated many times.

Technical note: For discrete outcomes x_i with probabilities p_i, E[X] = sum p_i x_i. A positive hit rate alone is not enough; payoff size matters.

Finance example: A strategy can win 70 percent of the time and still lose money if the average loss is much larger than the average gain.


## Code Lab 1: A reusable return function

In [ ]:
def simple_return(buy_price: float, sell_price: float) -> float:
    return sell_price / buy_price - 1

print(round(simple_return(100, 105), 4))
print(round(simple_return(50, 45), 4))


## Code Lab 2: Expected value across strategies

In [ ]:
import pandas as pd

strategies = pd.DataFrame(
    {
        "name": ["balanced", "high_hit_rate_bad_payoff", "convex_payoff"],
        "win_prob": [0.6, 0.8, 0.3],
        "win_payoff": [2.0, 1.0, 5.0],
        "loss_prob": [0.4, 0.2, 0.7],
        "loss_payoff": [-1.0, -5.0, -1.0],
    }
)
strategies["expected_value"] = (
    strategies["win_prob"] * strategies["win_payoff"]
    + strategies["loss_prob"] * strategies["loss_payoff"]
)
print(strategies[["name", "expected_value"]])


## Code Lab 3: Small simulation

In [ ]:
import numpy as np

rng = np.random.default_rng(42)
simulated = rng.choice([2.0, -1.0], size=20, p=[0.6, 0.4])
print(simulated)
print("Sample mean:", simulated.mean())


## Practice recap
- Why might a strategy with many winning trades still have negative expected value?
  Suggested answer: Because the losses may be much larger than the wins. Expected value depends on both probability and payoff magnitude.
- What does an if/else statement represent in a trading rule?
  Suggested answer: It represents a conditional action, such as buy if a signal exceeds a threshold and otherwise do nothing or reduce exposure.
- A setup has outcomes +4 with probability 0.25 and -1 with probability 0.75. What is expected value?
  Suggested answer: 0.25 * 4 + 0.75 * (-1) = 0.25.

## Interview drill
- Q: How is expected value different from the most likely outcome?
  A: Expected value is the average weighted outcome over many repetitions, while the most likely outcome is just the single highest-probability event.
- Q: Why do quants care about payoff asymmetry?
  A: Because good trading ideas are about the full payoff distribution, not just the percentage of winning trades.

## 4+ Hour Completion Roadmap

Use this minimum structure to turn today's notebook into a serious study session:

- Phase 1 (60 min): Read concept notes and rewrite the core idea from memory.
- Phase 2 (70 min): Complete and extend all code labs with at least one variation.
- Phase 3 (65 min): Do formula retrieval, scenario analysis, and error-log updates.
- Phase 4 (65 min): Write a mini memo and practice interview responses aloud.

## Formula Rewrite Drill

In [ ]:
import pandas as pd

formula_drill = pd.DataFrame(
    [
        {"formula": "E[X] = sum_i p_i x_i", "from_memory": "", "term_explanations": ""},
        {"formula": "Var(X) = E[(X - E[X])^2]", "from_memory": "", "term_explanations": ""},
        {"formula": "W_t = W_0 * product(1 + r_t)", "from_memory": "", "term_explanations": ""},
        {"formula": "Topic formula for: Functions, control flow, probability intuition, and expected value", "from_memory": "", "term_explanations": ""},
    ]
)
print(formula_drill)


## Real Market Data Lab (Useful From Day 1)

This section uses a local CSV snapshot of real market prices so the notebook remains reproducible.

Dataset: `curriculum/datasets/real_market_prices.csv`
Symbols: SPY, QQQ, TLT, GLD

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

data_path = Path("curriculum/datasets/real_market_prices.csv")
market = pd.read_csv(data_path, parse_dates=["date"]).sort_values(["date", "symbol"])

print("Rows:", len(market))
print("Date range:", market["date"].min().date(), "to", market["date"].max().date())
print("Symbols:", sorted(market["symbol"].unique()))
print(market.head())

prices = market.pivot(index="date", columns="symbol", values="close").dropna()
returns = prices.pct_change().dropna()

summary = pd.DataFrame(
    {
        "ann_return": returns.mean() * 252,
        "ann_vol": returns.std() * (252 ** 0.5),
        "max_drawdown": ((1 + returns).cumprod().div((1 + returns).cumprod().cummax()) - 1).min(),
    }
).sort_index()
print("\nRisk/return summary:")
print(summary.round(4))

cum = (1 + returns).cumprod()
cum.plot(figsize=(10, 5), title="Cumulative growth (base 1.0)")
plt.ylabel("Growth of $1")
plt.tight_layout()
plt.show()


## Real-World Takeaway Prompt

Write 5-8 lines for today's topic (Functions, control flow, probability intuition, and expected value):

1. Which symbol looked most volatile and why that matters.
2. Which pair looked most diversifying.
3. One realistic portfolio/risk decision you could make from this table.

## Interview Question + Python Solution Drill

Use this format for interview preparation:

1. State the question clearly.
2. Explain your approach in plain language.
3. Write Python code on real market data.
4. Interpret one risk caveat in words.

**Suggested data-source ladder**
- Source 1: yfinance pull (fresh market data)
- Source 2: Robinhood-style CSV export (if available locally)
- Source 3: local snapshot `curriculum/datasets/real_market_prices.csv` (reproducible fallback)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

prices = None
source_used = ""
USE_YFINANCE = False  # set True when you want a live market pull

try:
    import yfinance as yf
except ImportError:
    yf = None

if USE_YFINANCE and yf is not None:
    downloaded = yf.download(
        ["SPY", "QQQ", "TLT", "GLD"],
        period="2y",
        interval="1d",
        auto_adjust=True,
        progress=False,
    )
    if not downloaded.empty:
        if isinstance(downloaded.columns, pd.MultiIndex):
            if "Close" in downloaded.columns.get_level_values(0):
                prices = downloaded["Close"].copy()
            elif "Adj Close" in downloaded.columns.get_level_values(0):
                prices = downloaded["Adj Close"].copy()
        else:
            prices = downloaded.rename(columns=str.upper)
        source_used = "yfinance"

robinhood_export = Path("curriculum/datasets/robinhood_export.csv")
if prices is None and robinhood_export.exists():
    rh = pd.read_csv(robinhood_export, parse_dates=["date"])
    prices = rh.pivot(index="date", columns="symbol", values="close").sort_index()
    source_used = "robinhood_export_csv"

if prices is None:
    local = pd.read_csv(
        Path("curriculum/datasets/real_market_prices.csv"),
        parse_dates=["date"],
    )
    prices = local.pivot(index="date", columns="symbol", values="close").sort_index()
    source_used = "local_snapshot_csv"

prices = prices.dropna(how="all").ffill().dropna()
returns = prices.pct_change().dropna()
log_returns = np.log(prices / prices.shift(1)).dropna()

annualized_return = returns.mean() * 252
annualized_vol = returns.std() * np.sqrt(252)
sharpe_proxy = (annualized_return - 0.03) / annualized_vol

print("Source used:", source_used)
print("\nAnnualized return:")
print(annualized_return.round(4))
print("\nAnnualized volatility:")
print(annualized_vol.round(4))
print("\nSharpe proxy (rf=3%):")
print(sharpe_proxy.round(3))
print("\nRecent log returns:")
print(log_returns.tail().round(4))


## Scenario Analysis Drill

In [ ]:
import pandas as pd

scenarios = pd.DataFrame(
    [
        {"scenario": "base_case", "assumption": "", "expected_effect": ""},
        {"scenario": "bull_case", "assumption": "", "expected_effect": ""},
        {"scenario": "stress_case", "assumption": "", "expected_effect": ""},
    ]
)
print("Topic:", 'Functions, control flow, probability intuition, and expected value')
print(scenarios)


## Closed-Book Retrieval and Error Log

In [ ]:
import pandas as pd

retrieval_scorecard = pd.DataFrame(
    [
        {"prompt": "Explain today's concept in 3 lines", "score_0_to_5": None, "notes": ""},
        {"prompt": "Write one key formula without notes", "score_0_to_5": None, "notes": ""},
        {"prompt": "Give one realistic failure mode", "score_0_to_5": None, "notes": ""},
        {"prompt": "Connect to one trading/risk decision", "score_0_to_5": None, "notes": ""},
    ]
)

error_log = pd.DataFrame(
    [
        {
            "concept": "",
            "mistake": "",
            "correction": "",
            "next_review_date": "",
        }
    ]
)
print("Retrieval scorecard template:")
print(retrieval_scorecard)
print("\nError log template:")
print(error_log)


## Final 30-Minute Deliverable

Write a short memo (150-250 words) with this structure:

1. Core idea of Functions, control flow, probability intuition, and expected value in plain language.
2. One technical detail or formula and why it matters.
3. One practical quant use case.
4. One limitation or failure mode and how you would detect it.